# 05 - Asset Regime Analysis

This notebook analyzes regime detection **per asset** to understand:
1. Which algorithms perform well during which time periods
2. Correlations between assets across different regimes
3. Why the benchmark selects certain algorithms during certain periods

**Key insight**: Instead of applying regime detection to the benchmark (which is a composite), we apply it to each individual algorithm to find patterns like:
- "Algorithms A and B do well during period X"
- "Algorithms X and Y do well during period Y"
- This explains why the benchmark picks one or another

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

from src.analysis.regime_detector import (
    RegimeDetector, RegimeMethod,
    REGIME_EXPANSION, REGIME_SLOWDOWN, REGIME_RECESSION, REGIME_RECOVERY,
    INVESTMENT_CLOCK_REGIMES
)

# Configurar
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

DATA_PATH = Path('../data')
PROCESSED = DATA_PATH / 'processed'

## 1. Load Data

In [ ]:
# Load algorithm returns and benchmark weights
algo_returns = pd.read_parquet(PROCESSED / 'algo_returns.parquet')
benchmark_weights = pd.read_parquet(PROCESSED / 'benchmark_weights.parquet')

print(f"Algorithms: {algo_returns.shape[1]}")
print(f"Date range: {algo_returns.index.min()} to {algo_returns.index.max()}")
print(f"Total days: {len(algo_returns)}")

In [ ]:
# Select algorithms with significant data (at least 50% active)
min_active_ratio = 0.3
active_counts = (algo_returns != 0).sum()
valid_algos = active_counts[active_counts >= len(algo_returns) * min_active_ratio].index.tolist()

print(f"Algorithms with >= {min_active_ratio:.0%} active days: {len(valid_algos)}")

# Further filter: those with meaningful returns
algo_stats = algo_returns[valid_algos].describe().T
algo_stats['active_ratio'] = (algo_returns[valid_algos] != 0).sum() / len(algo_returns)
algo_stats['total_return'] = (1 + algo_returns[valid_algos]).prod() - 1

# Select top performing algos for detailed analysis
top_algos = algo_stats.nlargest(20, 'total_return').index.tolist()
print(f"\nTop 20 algorithms by total return: {top_algos[:5]}...")

## 2. Define Time Periods Based on Market Conditions

Instead of applying regime detection to a single aggregate series, we'll:
1. Calculate aggregate market conditions (using average of all algorithms)
2. Identify distinct time periods
3. Analyze which algorithms perform well in each period

In [ ]:
# Create market proxy: average return of active algorithms
active_mask = algo_returns != 0
market_returns = algo_returns.where(active_mask).mean(axis=1).fillna(0)

print(f"Market proxy - Mean return: {market_returns.mean():.4%}")
print(f"Market proxy - Volatility: {market_returns.std() * np.sqrt(252):.2%}")

In [ ]:
# Detect regimes on market proxy
detector = RegimeDetector(method=RegimeMethod.FUZZY, smoothing_window=10)
market_regimes = detector.detect(market_returns)

# Regime distribution
regime_counts = market_regimes.value_counts()
print("\nMarket Regime Distribution:")
for regime in INVESTMENT_CLOCK_REGIMES:
    count = regime_counts.get(regime, 0)
    pct = count / len(market_regimes) * 100
    print(f"  {regime:12}: {count:4d} days ({pct:5.1f}%)")

In [ ]:
# Visualize market regimes
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Equity curve with regime coloring
ax1 = axes[0]
equity = (1 + market_returns).cumprod() * 100
ax1.plot(equity.index, equity.values, 'k-', linewidth=1.5, alpha=0.8)

regime_colors = {
    REGIME_EXPANSION: 'lightgreen',
    REGIME_SLOWDOWN: 'khaki',
    REGIME_RECESSION: 'lightcoral',
    REGIME_RECOVERY: 'lightblue',
}

for regime, color in regime_colors.items():
    mask = market_regimes == regime
    if mask.any():
        ax1.fill_between(
            equity.index, equity.min() * 0.95, equity.max() * 1.05,
            where=mask.reindex(equity.index).fillna(False),
            alpha=0.4, color=color, label=regime.capitalize()
        )

ax1.set_ylabel('Market Proxy Equity', fontsize=12)
ax1.set_title('Market Regimes (Fuzzy Logic Investment Clock)', fontsize=14)
ax1.legend(loc='upper left', fontsize=10)

# Rolling volatility
ax2 = axes[1]
vol = market_returns.rolling(21).std() * np.sqrt(252)
ax2.fill_between(vol.index, 0, vol.values, alpha=0.4, color='blue')
ax2.axhline(vol.median(), color='red', linestyle='--', label=f'Median: {vol.median():.1%}')
ax2.set_ylabel('21d Rolling Volatility', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Identify Distinct Time Periods

Split the time series into consecutive regime periods to analyze algorithm performance.

In [ ]:
def identify_regime_periods(regimes: pd.Series, min_duration: int = 10) -> pd.DataFrame:
    """
    Identify consecutive periods of the same regime.
    
    Returns DataFrame with start, end, regime, duration.
    """
    periods = []
    current_regime = None
    start_date = None
    
    for date, regime in regimes.items():
        if regime != current_regime:
            if current_regime is not None:
                periods.append({
                    'start': start_date,
                    'end': prev_date,
                    'regime': current_regime,
                    'duration': (prev_date - start_date).days + 1
                })
            current_regime = regime
            start_date = date
        prev_date = date
    
    # Add last period
    if current_regime is not None:
        periods.append({
            'start': start_date,
            'end': prev_date,
            'regime': current_regime,
            'duration': (prev_date - start_date).days + 1
        })
    
    df = pd.DataFrame(periods)
    
    # Filter by minimum duration
    df = df[df['duration'] >= min_duration].reset_index(drop=True)
    df['period_id'] = range(len(df))
    
    return df

regime_periods = identify_regime_periods(market_regimes, min_duration=15)
print(f"Identified {len(regime_periods)} distinct regime periods (min 15 days):")
print(regime_periods.head(10))

In [ ]:
# Summary by regime
period_summary = regime_periods.groupby('regime').agg({
    'period_id': 'count',
    'duration': ['mean', 'median', 'sum']
}).round(1)
period_summary.columns = ['n_periods', 'avg_duration', 'median_duration', 'total_days']
period_summary = period_summary.sort_values('total_days', ascending=False)
print("\nPeriod Summary by Regime:")
print(period_summary)

## 4. Analyze Algorithm Performance by Regime Period

For each regime period, calculate which algorithms performed best.

In [ ]:
def calculate_algo_performance_in_period(returns: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.Series:
    """
    Calculate total return for each algorithm in a period.
    """
    period_returns = returns.loc[start:end]
    total_returns = (1 + period_returns).prod() - 1
    
    # Also calculate Sharpe-like metric
    mean_ret = period_returns.mean()
    std_ret = period_returns.std()
    sharpe = np.where(std_ret > 0, mean_ret / std_ret * np.sqrt(252), 0)
    
    return pd.DataFrame({
        'total_return': total_returns,
        'sharpe': sharpe,
        'n_active_days': (period_returns != 0).sum()
    })

# Calculate performance for each period
period_performances = []

for _, period in regime_periods.iterrows():
    perf = calculate_algo_performance_in_period(
        algo_returns[valid_algos], 
        period['start'], 
        period['end']
    )
    perf['period_id'] = period['period_id']
    perf['regime'] = period['regime']
    perf['algo_id'] = perf.index
    period_performances.append(perf.reset_index(drop=True))

all_performances = pd.concat(period_performances, ignore_index=True)
print(f"Calculated performance for {len(all_performances)} (algo x period) combinations")

In [ ]:
# Pivot to get algo x period matrix of returns
perf_matrix = all_performances.pivot(
    index='algo_id', 
    columns='period_id', 
    values='total_return'
).fillna(0)

print(f"Performance matrix shape: {perf_matrix.shape}")
print(f"  {perf_matrix.shape[0]} algorithms x {perf_matrix.shape[1]} periods")

## 5. Find Algorithm Clusters Based on Period Performance

Cluster algorithms by their performance patterns across periods. This reveals:
- Groups of algorithms that perform similarly across time
- Which algorithms are "regime specialists"

In [ ]:
# Normalize performance matrix for clustering
scaler = StandardScaler()
perf_scaled = scaler.fit_transform(perf_matrix.values)

# Hierarchical clustering
n_clusters = 6
clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
cluster_labels = clustering.fit_predict(perf_scaled)

perf_matrix['cluster'] = cluster_labels
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print(f"Algorithm clusters:")
for cluster, count in cluster_counts.items():
    print(f"  Cluster {cluster}: {count} algorithms")

In [ ]:
# Analyze each cluster's performance pattern
cluster_profiles = []

for cluster in range(n_clusters):
    cluster_algos = perf_matrix[perf_matrix['cluster'] == cluster].index.tolist()
    
    # Average performance by regime
    cluster_perf = all_performances[all_performances['algo_id'].isin(cluster_algos)]
    
    regime_perf = cluster_perf.groupby('regime')['total_return'].agg(['mean', 'std']).round(4)
    
    # Find best regime for this cluster
    best_regime = regime_perf['mean'].idxmax()
    worst_regime = regime_perf['mean'].idxmin()
    
    cluster_profiles.append({
        'cluster': cluster,
        'n_algos': len(cluster_algos),
        'best_regime': best_regime,
        'best_regime_return': regime_perf.loc[best_regime, 'mean'],
        'worst_regime': worst_regime,
        'worst_regime_return': regime_perf.loc[worst_regime, 'mean'],
        'expansion_return': regime_perf.loc[REGIME_EXPANSION, 'mean'] if REGIME_EXPANSION in regime_perf.index else 0,
        'recession_return': regime_perf.loc[REGIME_RECESSION, 'mean'] if REGIME_RECESSION in regime_perf.index else 0,
    })

cluster_profile_df = pd.DataFrame(cluster_profiles)
print("\nCluster Performance Profiles:")
print(cluster_profile_df.to_string())

In [ ]:
# Heatmap of cluster performance by regime
cluster_regime_matrix = all_performances.groupby(
    [all_performances['algo_id'].map(perf_matrix['cluster'].to_dict()), 'regime']
)['total_return'].mean().unstack(fill_value=0)

cluster_regime_matrix.index.name = 'Cluster'

# Reorder columns by Investment Clock order
col_order = [r for r in INVESTMENT_CLOCK_REGIMES if r in cluster_regime_matrix.columns]
cluster_regime_matrix = cluster_regime_matrix[col_order]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    cluster_regime_matrix * 100,  # Convert to percentage
    annot=True, fmt='.1f', cmap='RdYlGn', center=0,
    cbar_kws={'label': 'Avg Return (%)'},
    ax=ax
)
ax.set_title('Algorithm Cluster Performance by Market Regime', fontsize=14)
ax.set_xlabel('Market Regime', fontsize=12)
ax.set_ylabel('Algorithm Cluster', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis by Time Period

Find which algorithms are correlated within each regime (they move together).

In [ ]:
# Calculate correlation matrix for each regime
regime_correlations = {}

for regime in INVESTMENT_CLOCK_REGIMES:
    regime_mask = market_regimes == regime
    regime_returns = algo_returns.loc[regime_mask, top_algos]  # Use top algos for visualization
    
    if len(regime_returns) > 30:  # Need enough data
        corr_matrix = regime_returns.corr()
        regime_correlations[regime] = corr_matrix
        
        # Summary stats
        upper_tri = np.triu_indices_from(corr_matrix.values, k=1)
        upper_vals = corr_matrix.values[upper_tri]
        print(f"{regime:12}: mean_corr={np.mean(upper_vals):.3f}, "
              f"median={np.median(upper_vals):.3f}, n_days={len(regime_returns)}")

In [ ]:
# Visualize correlation changes across regimes
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (regime, corr) in enumerate(regime_correlations.items()):
    ax = axes[i]
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, cmap='coolwarm', center=0,
        vmin=-1, vmax=1, ax=ax,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f'{regime.capitalize()} Regime Correlations', fontsize=12)
    ax.tick_params(axis='both', labelsize=6)

plt.tight_layout()
plt.show()

## 7. Identify "Regime Specialists"

Find algorithms that significantly outperform in specific regimes.

In [ ]:
# Calculate average performance per algorithm per regime
algo_regime_perf = all_performances.groupby(['algo_id', 'regime'])['total_return'].mean().unstack(fill_value=0)

# Calculate performance differential (how much better in best regime vs overall)
algo_regime_perf['best_regime'] = algo_regime_perf[INVESTMENT_CLOCK_REGIMES].idxmax(axis=1)
algo_regime_perf['best_return'] = algo_regime_perf[INVESTMENT_CLOCK_REGIMES].max(axis=1)
algo_regime_perf['avg_return'] = algo_regime_perf[INVESTMENT_CLOCK_REGIMES].mean(axis=1)
algo_regime_perf['regime_alpha'] = algo_regime_perf['best_return'] - algo_regime_perf['avg_return']

# Filter for significant regime specialists
specialists = algo_regime_perf[algo_regime_perf['regime_alpha'] > 0.02].copy()
specialists = specialists.sort_values('regime_alpha', ascending=False)

print(f"Found {len(specialists)} regime specialists (alpha > 2%):")
print("\nTop 20 Regime Specialists:")
print(specialists.head(20)[['best_regime', 'best_return', 'avg_return', 'regime_alpha']].round(4))

In [ ]:
# Count specialists by regime
specialist_counts = specialists['best_regime'].value_counts()
print("\nSpecialists by Preferred Regime:")
for regime in INVESTMENT_CLOCK_REGIMES:
    count = specialist_counts.get(regime, 0)
    print(f"  {regime:12}: {count} algorithms")

In [ ]:
# Visualize specialists
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, regime in enumerate(INVESTMENT_CLOCK_REGIMES):
    ax = axes[i]
    regime_specialists = specialists[specialists['best_regime'] == regime].head(10)
    
    if len(regime_specialists) > 0:
        # Bar chart of top specialists
        y_pos = np.arange(len(regime_specialists))
        ax.barh(y_pos, regime_specialists['regime_alpha'] * 100, color=regime_colors.get(regime, 'gray'))
        ax.set_yticks(y_pos)
        ax.set_yticklabels([str(a)[:15] for a in regime_specialists.index], fontsize=8)
        ax.set_xlabel('Regime Alpha (%)', fontsize=10)
    
    ax.set_title(f'{regime.capitalize()} Specialists (Top 10)', fontsize=12)

plt.tight_layout()
plt.show()

## 8. Benchmark Selection Analysis

Analyze how the benchmark's algorithm selection relates to regime timing.

In [ ]:
# Load benchmark weights if available
if benchmark_weights is not None and len(benchmark_weights) > 0:
    # Align dates
    common_dates = benchmark_weights.index.intersection(market_regimes.index)
    bw_aligned = benchmark_weights.loc[common_dates]
    regimes_aligned = market_regimes.loc[common_dates]
    
    print(f"Analyzing benchmark weights for {len(common_dates)} days")
    
    # Average weight per algorithm per regime
    regime_benchmark_weights = {}
    for regime in INVESTMENT_CLOCK_REGIMES:
        mask = regimes_aligned == regime
        if mask.sum() > 0:
            avg_weights = bw_aligned.loc[mask].mean()
            regime_benchmark_weights[regime] = avg_weights
    
    regime_weights_df = pd.DataFrame(regime_benchmark_weights)
    regime_weights_df = regime_weights_df.dropna(how='all')
    
    print(f"\nTop 10 algorithms by average weight in each regime:")
    for regime in INVESTMENT_CLOCK_REGIMES:
        if regime in regime_weights_df.columns:
            top_in_regime = regime_weights_df[regime].nlargest(5)
            print(f"\n{regime.upper()}:")
            for algo, weight in top_in_regime.items():
                print(f"  {algo}: {weight:.2%}")
else:
    print("Benchmark weights not available for regime analysis")

In [ ]:
# Correlation between benchmark weight and regime specialist status
if 'regime_weights_df' in dir() and len(regime_weights_df) > 0:
    # For each regime, check if benchmark favors specialists
    analysis_results = []
    
    for regime in INVESTMENT_CLOCK_REGIMES:
        if regime not in regime_weights_df.columns:
            continue
            
        regime_wts = regime_weights_df[regime]
        
        # Get algorithms that are specialists for this regime
        specialist_algos = specialists[specialists['best_regime'] == regime].index
        
        # Calculate average weight for specialists vs non-specialists
        common_algos = regime_wts.index.intersection(specialist_algos)
        other_algos = regime_wts.index.difference(specialist_algos)
        
        if len(common_algos) > 0 and len(other_algos) > 0:
            specialist_weight = regime_wts.loc[common_algos].mean()
            other_weight = regime_wts.loc[other_algos].mean()
            
            analysis_results.append({
                'regime': regime,
                'n_specialists': len(common_algos),
                'specialist_avg_weight': specialist_weight,
                'non_specialist_avg_weight': other_weight,
                'weight_ratio': specialist_weight / other_weight if other_weight > 0 else np.inf
            })
    
    if analysis_results:
        analysis_df = pd.DataFrame(analysis_results)
        print("\nDoes the benchmark favor regime specialists?")
        print(analysis_df.round(4))
        print("\nWeight ratio > 1 means benchmark favors specialists for that regime")

## 9. Time Period Co-movement Analysis

Find pairs of algorithms that consistently perform well (or poorly) together in specific periods.

In [ ]:
# Calculate pairwise correlation of period-level performance
# (which algos have similar performance patterns across periods)
period_perf_corr = perf_matrix.drop('cluster', axis=1).T.corr()

print(f"Period-performance correlation matrix: {period_perf_corr.shape}")

# Find highly correlated pairs (move together across periods)
high_corr_pairs = []
for i, algo1 in enumerate(period_perf_corr.index):
    for j, algo2 in enumerate(period_perf_corr.columns):
        if i < j:  # Upper triangle only
            corr = period_perf_corr.loc[algo1, algo2]
            if abs(corr) > 0.7:  # High correlation threshold
                high_corr_pairs.append({
                    'algo1': algo1,
                    'algo2': algo2,
                    'correlation': corr,
                    'type': 'positive' if corr > 0 else 'negative'
                })

pairs_df = pd.DataFrame(high_corr_pairs)
pairs_df = pairs_df.sort_values('correlation', key=abs, ascending=False)

print(f"\nFound {len(pairs_df)} highly correlated algorithm pairs (|r| > 0.7):")
print(f"  Positive correlations: {(pairs_df['type'] == 'positive').sum()}")
print(f"  Negative correlations: {(pairs_df['type'] == 'negative').sum()}")

print("\nTop 20 most correlated pairs:")
print(pairs_df.head(20))

In [ ]:
# Visualize period performance correlation matrix (for top algorithms)
top_n = 30
top_algo_ids = perf_matrix.drop('cluster', axis=1).sum(axis=1).nlargest(top_n).index
top_corr = period_perf_corr.loc[top_algo_ids, top_algo_ids]

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(top_corr, dtype=bool))
sns.heatmap(
    top_corr, mask=mask, cmap='coolwarm', center=0,
    vmin=-1, vmax=1, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
    xticklabels=True, yticklabels=True
)
ax.set_title(f'Period Performance Correlation (Top {top_n} Algorithms)', fontsize=14)
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

## 10. Summary: Algorithm Groups by Time Period Performance

In [ ]:
# Create a comprehensive summary table
summary_data = []

for cluster in range(n_clusters):
    cluster_algos = perf_matrix[perf_matrix['cluster'] == cluster].index.tolist()
    cluster_perf = all_performances[all_performances['algo_id'].isin(cluster_algos)]
    
    # Performance by regime
    regime_returns = cluster_perf.groupby('regime')['total_return'].mean()
    
    # Best and worst regimes
    best_regime = regime_returns.idxmax() if len(regime_returns) > 0 else 'N/A'
    worst_regime = regime_returns.idxmin() if len(regime_returns) > 0 else 'N/A'
    
    # Sample algorithms from cluster
    sample_algos = cluster_algos[:3]
    
    summary_data.append({
        'cluster': cluster,
        'n_algorithms': len(cluster_algos),
        'best_regime': best_regime,
        'worst_regime': worst_regime,
        'expansion_ret': regime_returns.get(REGIME_EXPANSION, 0),
        'slowdown_ret': regime_returns.get(REGIME_SLOWDOWN, 0),
        'recession_ret': regime_returns.get(REGIME_RECESSION, 0),
        'recovery_ret': regime_returns.get(REGIME_RECOVERY, 0),
        'sample_algos': ', '.join([str(a)[:12] for a in sample_algos])
    })

summary_df = pd.DataFrame(summary_data)

print("=" * 100)
print("ALGORITHM CLUSTER SUMMARY BY REGIME PERFORMANCE")
print("=" * 100)
print()
print(summary_df.to_string())

# Interpretation
print("\n" + "=" * 100)
print("INTERPRETATION")
print("=" * 100)
print()
for _, row in summary_df.iterrows():
    print(f"Cluster {row['cluster']} ({row['n_algorithms']} algos):")
    print(f"  -> Best in: {row['best_regime'].upper()} regime")
    print(f"  -> Worst in: {row['worst_regime'].upper()} regime")
    print(f"  -> Sample: {row['sample_algos']}")
    print()

In [ ]:
# Save results
output_data = {
    'algo_regime_performance': algo_regime_perf,
    'cluster_assignments': perf_matrix[['cluster']],
    'regime_specialists': specialists,
    'correlated_pairs': pairs_df,
    'cluster_summary': summary_df
}

# Save to parquet
algo_regime_perf.to_parquet(PROCESSED / 'algo_regime_performance.parquet')
perf_matrix[['cluster']].to_parquet(PROCESSED / 'algo_period_clusters.parquet')

print("Results saved to:")
print("  - data/processed/algo_regime_performance.parquet")
print("  - data/processed/algo_period_clusters.parquet")

## Key Findings

This analysis reveals:

1. **Algorithm Clusters by Period Performance**: Algorithms naturally group into clusters based on when they perform well. Some clusters are "expansion specialists", others are "recession hedges".

2. **Regime Specialists**: Specific algorithms that significantly outperform during particular market regimes. These are candidates for regime-conditional allocation.

3. **Co-movement Patterns**: Algorithm pairs that consistently move together (or opposite) across different time periods. This informs diversification and hedge strategies.

4. **Benchmark Selection Logic**: Analysis of whether the benchmark systematically favors regime specialists during corresponding regimes, validating the investment clock approach.

### Implications for Meta-Allocator:
- Use regime detection to dynamically select algorithms
- Overweight "expansion specialists" during expansion, etc.
- Combine negatively correlated algorithm pairs for diversification
- The RL agent should learn to exploit these regime-performance patterns